# 🌊 Notebook 2: High-Water Mark to the Rescue

The **high-water mark (HWM)** is the highest log offset that has been safely replicated to a quorum of followers. Clients can only read entries with offset ≤ HWM. Anything past it is *tentative* and may be truncated if a new leader takes over.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 BETTER: HWM-gated reads

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Node:
    name: str
    log: List[str] = field(default_factory=list)

@dataclass
class Leader(Node):
    follower_match: Dict[str,int] = field(default_factory=dict)  # follower -> highest replicated offset (0-based +1)
    quorum: int = 2  # we need 2 nodes including leader

    def append(self, entry):
        self.log.append(entry)
        return len(self.log) - 1

    def ack(self, follower, offset):
        self.follower_match[follower] = max(self.follower_match.get(follower,-1), offset)

    @property
    def high_water_mark(self):
        # Sort all match offsets (leader counts as having everything)
        matches = sorted([len(self.log)-1] + list(self.follower_match.values()), reverse=True)
        # The Nth element where N = quorum is the highest offset replicated to >=quorum
        return matches[self.quorum - 1] if matches else -1

    def committed_view(self):
        return self.log[: self.high_water_mark + 1]

leader = Leader('L')
f1, f2 = Node('f1'), Node('f2')

def replicate(entry, delivered_to):
    off = leader.append(entry)
    for f in delivered_to:
        f.log.append(entry)
        leader.ack(f.name, off)
    print(f'append {entry!r} off={off} hwm={leader.high_water_mark} client_view={leader.committed_view()}')

replicate('A', [f1, f2])    # quorum -> committed
replicate('B', [f1])        # only one follower -> still committed (f1 + leader = quorum)
replicate('C', [])          # no followers -> NOT committed


Even though the leader has `C` in its local log, the HWM stays at offset 1 (`B`). Clients never see `C`, so they never make a decision based on it.

## 💥 Now crash the leader

In [ ]:
client_view_before_crash = leader.committed_view()
print('client saw before crash:', client_view_before_crash)

# Promote follower with longest log
candidates = [f1, f2]
new_leader = max(candidates, key=lambda f: len(f.log))
print('new leader log:', new_leader.log)
print('all committed entries survived?', all(e in new_leader.log for e in client_view_before_crash))


✅ Every entry the client *saw* survives the failover. The uncommitted `C` is silently discarded — which is exactly what we want.

## 🧠 Takeaways

- The HWM is just **the offset replicated to a quorum**, recomputed as acks come in.
- Clients read **at or below** the HWM.
- Followers learn the HWM from the leader and may truncate anything beyond it after a leadership change.
- This is the core of how Kafka, Raft, and Multi-Paxos avoid losing acknowledged writes.